In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

In [3]:
files_path = "../data/raw/archive"

data_path = Path(files_path)
csv_files = list(data_path.glob("*.csv"))
print(f"CSV files: {csv_files}")


def read_datasets(datasets):
    dataframes = {}
    for dataset in datasets:
        file = dataset.name
        dataframes[file] = pd.read_csv(dataset)
    return dataframes
        
datasets = read_datasets(csv_files)
datasets_names = datasets.keys()

for dataset_name in datasets_names:
    print(dataset_name)

CSV files: [WindowsPath('../data/raw/archive/olist_customers_dataset.csv'), WindowsPath('../data/raw/archive/olist_geolocation_dataset.csv'), WindowsPath('../data/raw/archive/olist_orders_dataset.csv'), WindowsPath('../data/raw/archive/olist_order_items_dataset.csv'), WindowsPath('../data/raw/archive/olist_order_payments_dataset.csv'), WindowsPath('../data/raw/archive/olist_order_reviews_dataset.csv'), WindowsPath('../data/raw/archive/olist_products_dataset.csv'), WindowsPath('../data/raw/archive/olist_sellers_dataset.csv'), WindowsPath('../data/raw/archive/product_category_name_translation.csv')]
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [4]:
##Olist Datasets Summary##
def datasets_summary(datasets_files, datasets):
    for dataset in datasets_files:
        dataset_name = dataset.name
        dataset_pd = datasets[dataset_name]
        columns_names = dataset_pd.columns
        columns_info = dataset_pd.dtypes

        rows, columns = dataset_pd.shape
        missing_values = dataset_pd.isna().sum()

        print(dataset_name)
        print(f"Columns names: {list(columns_names)}")
        print(columns_info)
        print(f"Rows: {rows}")
        print(f"Columns: {columns}\n")
        print("Missing values:")
        print(f"{missing_values}\n")
        print("Missing values percentage:")
        print((missing_values / len(dataset_pd)) * 100)

        ##Duplicates Check##
        duplicates_sum = dataset_pd.duplicated().sum()
        print(f"Total duplicates sums: {duplicates_sum}")

        print("-" * 80)
datasets_summary(csv_files, datasets)

olist_customers_dataset.csv
Columns names: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
Rows: 99441
Columns: 5

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Missing values percentage:
customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64
Total duplicates sums: 0
--------------------------------------------------------------------------------
olist_geolocation_dataset.csv
Columns names: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
geol

In [5]:
##Duplicates analysis##
def duplicates_check(dataset_name, datasets):
    dataset = datasets[dataset_name]
    dataset.head()
    duplicates = dataset.duplicated()
    print(f"Total number of duplicates: {duplicates.sum()}")
    display(dataset[duplicates])
duplicates_check(dataset_name="olist_orders_dataset.csv", datasets=datasets)

## Based on this the duplicates seem to be associated only with the same order location so there is no need for removal
## No other duplicates were noticed in any of the datasets


Total number of duplicates: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [6]:
##Missing values analysis
def missing_orders_check(dataset_name, datasets):
    dataset = datasets[dataset_name]
    missing_values = dataset[dataset.isna().any(axis=1)]
    order_status_missing = list(missing_values["order_status"].unique())
    total_count = 0
    for order_status in order_status_missing:
        order_count = (missing_values["order_status"] == order_status).sum()
        total_count += order_count
        print(f"Total count of {order_status} orders with missing values: {order_count}")
    print(f"Total missing count: {total_count}")

    delivered_status_missing = missing_values[missing_values["order_status"] == "shipped"]
    print ((delivered_status_missing["order_delivered_customer_date"].isna()).sum())

    
missing_orders_check(dataset_name="olist_orders_dataset.csv", datasets=datasets)

#The missing values analysis for the "olist_orders_dataset.csv" showed that several rows have missing data - all missing values are related to the timestamp columns such as:
# order_delivered_customer_date which rows perecentage of missing relative to the total rows length is around 3%. Surprisingly even orders with the delivery status delivered have missing timestamps information, suggesting possible 
# data missing. The top category with the most amount of rows with missing values based on order status are the "shipped" orders. In addition all of the shipped order status had the order_delivered_customer_date, which makes sense because the order has not been delivered yet.

Total count of invoiced orders with missing values: 314
Total count of shipped orders with missing values: 1107
Total count of processing orders with missing values: 301
Total count of unavailable orders with missing values: 609
Total count of canceled orders with missing values: 619
Total count of delivered orders with missing values: 23
Total count of created orders with missing values: 5
Total count of approved orders with missing values: 2
Total missing count: 2980
1107


In [7]:
#Missing review values
def review_missing_values(dataset_name, datasets):
    dataset = datasets[dataset_name]
    missing_values = dataset[dataset.isna().any(axis=1)]
    missing_titles = missing_values["review_comment_title"].isna()
    missing_comments_messages = missing_values["review_comment_message"].isna()
    display (missing_values)
    display(f"Total amount of missing review comment titles: {int(missing_titles.sum())}")
    display(f"Total amount of missing review comment messages: {int(missing_comments_messages.sum())}")
review_missing_values(dataset_name="olist_order_reviews_dataset.csv", datasets=datasets)
#All reviews havea score, while most of the comments titles (~80%) are missing, and around 58% of the comments messages are also missing.
#I think this is not a big deal because review score is always present

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


'Total amount of missing review comment titles: 87656'

'Total amount of missing review comment messages: 58247'

In [8]:
def missing_products_check(dataset_name, datasets):
    dataset = datasets[dataset_name]
    missing_values = dataset[dataset.isna().any(axis=1)]
    return missing_values
missing_products_check(dataset_name="olist_products_dataset.csv", datasets=datasets)
#There are approximatelly 611 products with missing "product_category_name, product_name_lenght, product_description_lenght, product_photos_qty"
#Even though its small amount of products, its necessary to check in the future if any people have actually ordered these products, and if not then 
#deleting the products might be justifyable.

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


### Establishing the tables connections

In [9]:
def find_key_column(datasets, target_dataset_name, connections_table):

    datasets_names = datasets.keys()
    
    target_dataset_pd = datasets[target_dataset_name]
    target_columns = list(target_dataset_pd.columns)

    for dataset_name in datasets_names:
        if dataset_name == target_dataset_name:
            continue
        dataset_pd = datasets[dataset_name]
        dataset_columns = list(dataset_pd.columns)

        shared_columns = []

        for target_column in target_columns:
            if target_column in dataset_columns:
                shared_columns.append(target_column)
        if shared_columns:
            connections_table.loc[dataset_name, target_dataset_name] = shared_columns
    return connections_table


connections_table = pd.DataFrame(
index=datasets_names,
columns=datasets_names
)
            
for dataset_name in list(datasets.keys()):
    find_key_column(datasets=datasets, target_dataset_name=dataset_name, connections_table=connections_table)

connections_table

,olist_customers_dataset.csv,olist_geolocation_dataset.csv,olist_orders_dataset.csv,olist_order_items_dataset.csv,olist_order_payments_dataset.csv,olist_order_reviews_dataset.csv,olist_products_dataset.csv,olist_sellers_dataset.csv,product_category_name_translation.csv
olist_customers_dataset.csv,NaN,NaN,[customer_id],NaN,NaN,NaN,NaN,NaN,NaN
olist_geolocation_dataset.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
olist_orders_dataset.csv,[customer_id],NaN,NaN,[order_id],[order_id],[order_id],NaN,NaN,NaN
olist_order_items_dataset.csv,NaN,NaN,[order_id],NaN,[order_id],[order_id],[product_id],[seller_id],NaN
olist_order_payments_dataset.csv,NaN,NaN,[order_id],[order_id],NaN,[order_id],NaN,NaN,NaN
olist_order_reviews_dataset.csv,NaN,NaN,[order_id],[order_id],[order_id],NaN,NaN,NaN,NaN
olist_products_dataset.csv,NaN,NaN,NaN,[product_id],NaN,NaN,NaN,NaN,[product_category_name]
olist_sellers_dataset.csv,NaN,NaN,NaN,[seller_id],NaN,NaN,NaN,NaN,NaN
product_category_name_translation.csv,NaN,NaN,NaN,NaN,NaN,NaN,[product_category_name],NaN,NaN


In [10]:
#Check geolocation columns
geo_dataset = datasets["olist_geolocation_dataset.csv"].sort_values(by="geolocation_zip_code_prefix", ascending=True)
#display(geo_dataset.head())
#display(geo_dataset.shape)
display(geo_dataset[geo_dataset["geolocation_zip_code_prefix"] == 1021].head(1))

customers_dataset = datasets["olist_customers_dataset.csv"].sort_values(by="customer_zip_code_prefix", ascending=True)
#display(customers_dataset.head())
#display(customers_dataset.shape)
display(customers_dataset[customers_dataset["customer_zip_code_prefix"] == 1021].head(1))

sellers_dataset = datasets["olist_sellers_dataset.csv"].sort_values(by="seller_zip_code_prefix", ascending=True)
#display(sellers_dataset.head())
#display(sellers_dataset.shape)
display(sellers_dataset[sellers_dataset["seller_zip_code_prefix"] == 1021].head(1))

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
702,1021,-23.544601,-46.63254,sao paulo,SP


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
55157,b01592e974c5b73b5adc1cafc1ebb7e9,2d51fb14470de565e7930d56becb3f45,1021,sao paulo,SP


,seller_id,seller_zip_code_prefix,seller_city,seller_state
599,e0487761face83d64fcada2408959a36,1021,sao paulo,SP


1. Based on the "connections table" which represent shared columns in the datasets, we can see that customers dataset is connected to the orders via the customer_id.
2. Geolocation dataset seems to not have any connecting columns, but the issue might be that the columns names are not exactly the same. But there are zipcode prefix, customer_city and customer state columns in both sellers and customers information dataset. Further analysis has confirmed it.
3. Orders dataset is connected to the customers through customer id. At the same time supplementary order information, such as order payment, review and order items are connected through the order_id key.
4. order_id seems to be the key column which connects to various other datasets 
5. customer_id is what connects a specific customer to the order dataset

In [11]:
#customer_id analysis
display(f"customer_id duplicate check: {customers_dataset["customer_id"].duplicated().sum()}") #customer_id creates everytime new order is created
display(f"customer_unique_id duplicate check: {customers_dataset["customer_unique_id"].duplicated().sum()}") #there are duplicates
duplicated_dataset = customers_dataset[customers_dataset["customer_unique_id"].duplicated(keep=False)] #based on this we can see that even the same customer can have difference customer_id with multiple orders
print("-"*150)
print("Customers dataset containing only customers who have more than one customer_id")
display(duplicated_dataset.head())
print("-"*150)

orders_dataset = datasets["olist_orders_dataset.csv"]

#Perfrom connection check by customer_id and unique customer_id
print("Results for the customer with unique id and repeated ids")
display(customers_dataset[customers_dataset["customer_unique_id"] == "ded4351942c7fc292b88e5b090af2b46"])
display(orders_dataset[orders_dataset["customer_id"].isin(["6e4eb34e7f4d526a82726712aa17c02b", "f3e31afdae80581be48ce94e7b0f3366"])])

#Based on this we can see that the same customer with one unique_customer_id has multiple customer_id for each order

'customer_id duplicate check: 0'

'customer_unique_id duplicate check: 3345'

------------------------------------------------------------------------------------------------------------------------------------------------------
Customers dataset containing only customers who have more than one customer_id


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
92799,6e4eb34e7f4d526a82726712aa17c02b,ded4351942c7fc292b88e5b090af2b46,1005,sao paulo,SP
34298,f3e31afdae80581be48ce94e7b0f3366,ded4351942c7fc292b88e5b090af2b46,1005,sao paulo,SP
90060,410bf44fa819e660e2fcaea12c35af43,4092bf290aa60e4f2d4a0dcc2f38c954,1009,sao paulo,SP
70517,d4d6b4b9d85471149150ceda6dee1501,4092bf290aa60e4f2d4a0dcc2f38c954,1009,sao paulo,SP
15072,ac8cbf914049a7b20272a10f768ba6dd,91352e06066eef42b60b24049ccde4fd,1011,sao paulo,SP


------------------------------------------------------------------------------------------------------------------------------------------------------
Results for the customer with unique id and repeated ids


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
92799,6e4eb34e7f4d526a82726712aa17c02b,ded4351942c7fc292b88e5b090af2b46,1005,sao paulo,SP
34298,f3e31afdae80581be48ce94e7b0f3366,ded4351942c7fc292b88e5b090af2b46,1005,sao paulo,SP


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5136,ef7c5d4791639dc1862b7bb3bacb3d48,f3e31afdae80581be48ce94e7b0f3366,delivered,2018-08-06 11:32:42,2018-08-06 11:45:14,2018-08-09 14:44:00,2018-08-13 20:18:42,2018-09-03 00:00:00
48024,3e65d8271b054ca287c93d4b56c55386,6e4eb34e7f4d526a82726712aa17c02b,delivered,2018-04-07 14:16:30,2018-04-07 14:30:14,2018-04-10 19:12:10,2018-04-11 11:11:37,2018-04-25 00:00:00


product_id relathionship analysis

In [13]:
datasets["olist_order_payments_dataset.csv"].head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
